# 04 · Validate — coordination-geometry preservation, scheme comparison, docking, site pLDDT & MD

**Standard slot:** *validate (in silico).* **For Project 24 this is the benchmark:** the
**coordination-geometry preservation rate**, a **cofactor/scheme comparison** (bis-His heme vs
His/Met heme vs [4Fe-4S] vs Zn) `[extension]`, the cofactor-docking / site-pLDDT / caveated-MD
figures, and a **redox-tuning** discussion (D3 pt2).

Needs `results/campaign.csv` (+ `results/ranked.csv` from notebook 03).

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Coordination-geometry preservation — the headline figure
Distribution of coordination-geometry RMSD vs the 0.5 Å pass bar. The fraction left of the line is the
**preservation rate** — the metric that most distinguishes scaffolding tools and cofactor schemes.
(Numbers here are SYNTHETIC mock values; on Colab they come from real AF2 predictions with the
metal/cofactor placed by docking/superposition first.)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

camp = pd.read_csv("results/campaign.csv")
cut = 0.5

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.hist(camp["coordination_geom_rmsd"], bins=20)
ax.axvline(cut, color="k", ls="--", lw=1, label=f"pass < {cut} A")
ax.set_xlabel("coordination-geometry RMSD vs target scheme (A)  [SYNTHETIC]")
ax.set_ylabel("designs"); ax.set_title("Coordination-geometry preservation")
ax.legend(); plt.tight_layout()
plt.savefig("results/coordination_geometry_hist.png", dpi=150); plt.show()

rate = 100 * (camp["coordination_geom_rmsd"] <= cut).mean()
print(f"overall coordination-geometry preservation rate = {rate:.1f}%  [SYNTHETIC demo]")
print("REALITY CHECK: a good geometry rate is NOT an incorporation rate, let alone a function rate.")

## 2 · Cofactor / scheme comparison `[extension]`
Compare the preservation rate (and hit rate) across **cofactor schemes**. In the real campaign you run
the SAME pipeline for bis-His heme vs His/Met heme vs [4Fe-4S]-4Cys vs Cys2His2 Zn (a harder
coordination is harder to scaffold cleanly); here a single mock scheme is present, so this cell shows
the *shape* of the comparison you will populate on Colab.

In [ ]:
by_cofactor = (camp.assign(pass_geom=camp["coordination_geom_rmsd"] <= 0.5)
                   .groupby("cofactor")
                   .agg(n=("design_id", "size"),
                        geom_pass_rate=("pass_geom", "mean"),
                        mean_plddt_site=("plddt_site", "mean"),
                        fe_between_ligands_rate=("fe_between_ligands", "mean"))
                   .reset_index())
by_cofactor["geom_pass_rate"] = (100 * by_cofactor["geom_pass_rate"]).round(1)
by_cofactor["fe_between_ligands_rate"] = (100 * by_cofactor["fe_between_ligands_rate"]).round(1)
print("Cofactor/scheme comparison (populate with real schemes on Colab):")
print(by_cofactor.to_string(index=False))
print("\n[SYNTHETIC] On Colab: run the SAME pipeline for bis-His heme vs His/Met heme vs [4Fe-4S] vs Zn.")

## 3 · Cofactor docking + site pLDDT + caveated active-site MD (top candidates)
Docking (AutoDock Vina) checks the cofactor **fits and is oriented** — for heme, does the porphyrin
fit and does the Fe sit **between** the axial ligands? — *not* affinity, not incorporation, not
function. The **site pLDDT** is the trustworthy confidence (the global pLDDT can look great while the
coordinating atoms are misplaced). Short MD (OpenMM) checks the pocket doesn't drift — but **classical
metal/heme force fields are approximate**, so treat it as a weak, caveated proxy. Plot these for the
ranked survivors as orthogonal evidence.

In [ ]:
import matplotlib.pyplot as plt
# ranked.csv comes from the shared filter (fp.Design fields); vina_score/site pLDDT live in
# campaign.csv, so merge them back by design_id for the docking-vs-MD view.
try:
    ranked = pd.read_csv("results/ranked.csv")
    ranked = ranked.merge(camp[["design_id", "vina_score", "plddt_site", "fe_between_ligands"]],
                          on="design_id", how="left")
except FileNotFoundError:
    ranked = camp.copy()

top = ranked.head(min(20, len(ranked)))
fig, ax = plt.subplots(figsize=(5.4, 3.4))
sc = ax.scatter(top["vina_score"], top["md_rmsd"],
                c=top["coordination_geom_rmsd"] if "coordination_geom_rmsd" in top
                  else top["catalytic_geom_rmsd"], cmap="viridis")
ax.set_xlabel("Vina cofactor-fit score (more negative = better fit)  [SYNTHETIC]")
ax.set_ylabel("pocket MD RMSD (A) — CAVEATED metal-FF proxy  [SYNTHETIC]")
ax.set_title("Top candidates: cofactor fit vs pocket stability")
fig.colorbar(sc, label="coordination-geom RMSD (A)")
plt.tight_layout(); plt.savefig("results/docking_md.png", dpi=150); plt.show()
print("Lower-left + dark points (good fit, stable, good coordination geometry) are the best [SYNTHETIC].")
print("MD CAVEAT: classical metal/heme FF is approximate — weak proxy only; spectroscopy is the real test.")

## 4 · Honest hit-rate accounting + redox-tuning reasoning `[extension]`
Report N(pass all layers) / N(generated), and remind the reader of the field reality: even a good
preservation rate is **not** an incorporation rate, and incorporation is **not** function. Then reason
(qualitatively — **no fabricated numbers**) about **redox tuning**: how the **axial-ligand identity**
(His vs Met), **second-shell** residues, and **pocket polarity** would be expected to shift the heme
redox midpoint / O₂ affinity — the hard, valuable problem after coordination.

In [ ]:
n_total = len(camp)
try:
    ranked = pd.read_csv("results/ranked.csv")
    n_hits = int((ranked["layers_passed"] >= 3).sum())
except Exception:
    n_hits = int((camp["coordination_geom_rmsd"] <= 0.5).sum())
print("Hit-rate accounting [SYNTHETIC demo]:")
print(f"  generated             : {n_total}")
print(f"  pass all filter layers: {n_hits}  ({100*n_hits/max(n_total,1):.1f}%)")
print("\nREALITY CHECK: de novo metalloprotein hit rates are LOW, and coordination geometry does NOT")
print("guarantee cofactor incorporation, let alone the designed redox/O2 behaviour. Only spectroscopy decides.")
print("\nRedox-tuning reasoning [extension] (qualitative — NEVER fabricate a midpoint potential):")
print("  - axial-ligand identity (bis-His vs His/Met) shifts the midpoint and spin state;")
print("  - second-shell H-bonds / charges around the propionates tune the potential;")
print("  - pocket hydrophobicity/burial raises/lowers the potential; an open distal pocket -> O2 binding.")

## D3 (part 2) checklist
- [ ] Coordination-geometry preservation histogram (`results/coordination_geometry_hist.png`) + rate.
- [ ] Cofactor/scheme comparison table/figure (real schemes on Colab) `[extension]`.
- [ ] Cofactor-docking + site-pLDDT + caveated-MD figure on the ranked top set.
- [ ] Honest hit-rate accounting with the "geometry ≠ incorporation ≠ function" caveat stated.
- [ ] Redox-tuning reasoning written up (qualitative; no fabricated potentials) `[extension]`.

**Next:** `05_validation_plan.ipynb` — the spectroscopic-assay plan + cofactor titration + controls.